# Chapter 2: What Is an LLM?

[Read this chapter online](https://jackluu.io/book/section-1-foundations/ch02-what-is-an-llm/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch02-what-is-an-llm.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 2: What Is an LLM?

![The map with Module 1 highlighted](https://jackluu.io/book/assets/diagrams/ch02-where-we-are.png)

*Figure 2.1: What the whole system does, before we zoom in.*

Now that we have our environment set up from the previous chapter, we can start to look at our model (Figure 2.1). A Language Model (LM) can be thought of as a black box that understands what you put into it. But all it does is one very basic task.

In this chapter you will:

- Learn the single rule that drives all language models.
- See how models generate long answers one step at a time.
- Understand how reading data replaces human teachers.

> **Words to Know**
>
> - **Token**: one small piece of text, here a single character.
> - **Language Model**: a system that predicts the next token in a sequence.
> - **Autoregressive**: building an answer one piece at a time, feeding each new piece back in before predicting the next one.
> - **Parameters**: the numbers inside the model that adjust during training.

## Theory: The Next-Token Engine

Imagine you are texting a friend and you type: "I am so hungry, I could eat a". Before you finish, your phone suggests: **horse**. Or **pizza**.

![A text box with incomplete text points to a prediction box showing horse](https://jackluu.io/book/assets/diagrams/ch02-next-word.png)

*Figure 2.2: The core task of a language model is guessing what comes next.*

So your phone is doing **next-word prediction** (Figure 2.2). Based on all the words you have entered, it will try to predict what you will type next. A Large Language Model (LLM) does the same thing. Given a sequence of tokens, it will try to predict the next token. In the rest of this book, we will learn how to build a model that can do this well.

### How It Talks Back

So if it can only predict one token at a time, how does it generate essays? It generates a token, then adds that token to its output and generates another. It continues doing this until it has finished what it was trying to generate. This is known as **autoregressive generation**.

1. You type: `"What is the capital of France?"`
2. The model predicts the next token: `"Paris"`.
3. That output is added to the input: `"What is the capital of France? Paris"`.
4. It predicts the next token: `" is"`.
5. And repeats until it decides to stop.

![The output text fed back into the model as input](https://jackluu.io/book/assets/diagrams/ch02-autoregressive.png)

*Figure 2.3: In autoregressive generation, each predicted token is added to the input.*

The model never thinks about the whole answer at once (Figure 2.3). It keeps predicting the next piece, over and over, and the sentence builds up step by step.

### What is a Token?

The smallest unit that our model will work with is called a token. This could be a word or a portion of a word or just one character.

In this book we will be working with characters as our tokens. So each character including spaces and punctuation is considered a token. We choose to work with characters because our vocabulary size is very small (65 characters total in our Shakespeare dataset). And it's easy to follow what each character means. Real models will use a portion of words called subwords instead of just characters. But the math is the same.

### The Power of Self-Supervised Learning

When you want to make your model "Large", you give it millions of parameters. Parameters are just values in the model that we can adjust. We train our model by adjusting these values until it can generate decent responses.

There is a type of training where you don't need a teacher to tell the model if it's right or wrong. This is known as **self-supervised learning**.

Because we're working with language, we have the answer already. So if we have a sentence that says "Hello world", the model gets these practice questions automatically:

- See `H`, predict `e`.
- See `He`, predict `l`.
- See `Hel`, predict `l`.

![Text subsets predicting the next character](https://jackluu.io/book/assets/diagrams/ch02-self-supervised.png)

*Figure 2.4: The text itself provides the practice questions.*

Since no human-labeled data is needed, you can use millions of pages of raw text to train a model (Figure 2.4). It will learn the patterns from the information given.

## Try It: The Numbers Game

Let's see if we can predict the next character on actual data. Given the characters `"the "`, what do you think will come next in Shakespeare?

We created a small script that loops through all of our training data and counts every character that follows `"the "`.

```text
$ python src/examples/ch02_next_char.py
What follows 'the ' in Shakespeare?
's': 573 times
'w': 459 times
'c': 448 times
'p': 426 times
'm': 376 times
```

Our model will learn to do this as well, but instead of counting, it will use math to figure out which characters are most likely to come next.

> **In Business**
>
> You'd use that same principle if you wanted to teach an AI how to write emails for you that sound like your company's house style. It would learn from all your past correspondence, deducing which letters tend to follow which, and which words.
>

> **Watch Out**
>
> It is easy to think the model "understands" the text. All it does is calculate the most probable next token from patterns in its training data.

## Key Takeaways

- A language model predicts the next token.
- Autoregressive generation refers to feeding the model’s own outputs back as input.
- Tokens are segments of text; they could be characters or words.
- We use the raw text itself as the “answer key” in self-supervised learning.
- As you can see, everything here revolves around next-token prediction! The first stop on our map will cover the math and tools we use to create a language model.

## Check Your Understanding
1. What does a language model predict?
2. Why is it called "autoregressive"?
3. How does self-supervised learning differ from having a teacher grade the work?

## Assignment

> **Assignment 2: New word, new guess**
>
> **About 15 minutes. You change one word and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/examples/ch02_next_char.py` as it comes. Cut and paste the five letters and counts.
> 2. **Change one word.** Locate the line `prefix = "the "`. The prefix is what the program searches for. Replace `the` with `my` and keep the space before the closing quotation mark, so that your line looks like `prefix = "my "`. Run the program again.
> 3. **Compare.** What letter is first now? Which letters appear in both groups? Think of one word Shakespeare could write after `my` that starts with the new first letter.
> 4. **Explain it.** In two or three sentences a manager could understand, explain why the most common next letter changed when you changed one word.
>
> **Hand in:** both lists of five letters and counts, your answers to step 3, and your sentences.

## Further Reading

**One model, many jobs, no retraining.** The researchers wanted to know if a model trained exclusively to predict the next word would learn capabilities nobody had trained it for. Because it was trained on such a broad, diverse swath of web pages, it started answering questions, summarizing, and translating with zero task-specific training when given a prompt that made those tasks obvious. That answered the architectural question of how to build a system for text generation in favor of a decoder architecture that predicts the next token. That's the model we cover in Chapter 10.

**Memory, before attention.** Sequential models read a sentence one word at a time and had mostly forgotten it by the end, due to the vanishing gradient as error flowed back through time steps. This work introduced a cell with gates that controlled what to remember, what to forget and what to output. Gate cells like this powered essentially all serious language applications for two decades. The cell poses the question of what the model should remember from earlier in the sentence. Attention later modeled the same concept using a different mechanism.


Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735

Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). *Language models are unsupervised multitask learners* [Technical report]. OpenAI. https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

---

### `src/examples/ch02_next_char.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch02_next_char.py"   # a cell has none, and the file uses it to find the text

"""Shows what character follows the string 'the ' in Shakespeare."""
import os
import sys
from collections import Counter

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

# Read the dataset
data_path = os.path.join(os.path.dirname(__file__), "..", "data", "shakespeare.txt")
with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

# Find what follows "the "
prefix = "the "
next_chars = []
for i in range(len(text) - len(prefix)):
    if text[i:i+len(prefix)].lower() == prefix:
        next_chars.append(text[i+len(prefix)])

# Count and print the top 5
counts = Counter(next_chars)
print(f"What follows '{prefix}' in Shakespeare?")
for char, count in counts.most_common(5):
    display_char = repr(char) if char == ' ' or char == '\n' else char
    print(f"'{display_char}': {count} times")